In [4]:
import brainsss
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
#from sklearn.cluster import AgglomerativeClustering
import scipy
from scipy.cluster.hierarchy import dendrogram
from scipy.cluster.hierarchy import fcluster
from scipy.cluster import hierarchy
from scipy.signal import butter, sosfiltfilt, filtfilt, freqz
from scipy import signal
import matplotlib as mpl
from matplotlib.pyplot import cm
import random
from scipy.stats import sem
import time
import h5py
import ants
import nibabel as nib
import matplotlib
from scipy.ndimage import gaussian_filter1d,gaussian_filter
import pickle
from skimage import io, filters
import glob
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph
import psutil
from brainsss import brain_utils
from xml.etree import ElementTree as ET

In [13]:
fly_num = 'fly_209'
func_path = f'/oak/stanford/groups/trc/data/Ilana/2P/data/{fly_num}/func_0'

xml_file = 'imaging/functional.xml'

xml_dir = os.path.join(func_path, xml_file)



In [14]:
tree = ET.parse(xml_dir)
root = tree.getroot()

In [26]:
sequences = root.findall('Sequence')
# sequences
len(sequences)

3384

In [37]:
first_frame_len = len(sequences[0].findall('Frame'))
# first_frame_len
timestamps=[]

In [38]:
for sequence_i in range(len(sequences)):
    sequence = sequences[sequence_i]
    frames = sequence.findall('Frame')
    #skip remaining sequences if ended early
    if len(frames) != first_frame_len:
        print(f'sequence # {sequence_i} did not complete z-series => ending')
        sequence_length = sequence_i #want it to be length to previous sequence value since that was the last complete one => 0 indexing helps
    else:
        for frame in frames:
            filename = frame.findall('File')[0].get('filename')
            time = float(frame.get('relativeTime'))
            timestamps.append(time)
        sequence_length = len(sequences)
timestamps = np.multiply(timestamps, 1000)



In [39]:
np.shape(timestamps)

(165816,)

In [40]:
if len(sequences) > 1:
    timestamps = np.reshape(timestamps, (sequence_length, first_frame_len))
else:
    timestamps = np.reshape(timestamps, (first_frame_len, sequence_length))



In [41]:
np.shape(timestamps)

(3384, 49)

In [73]:
T=(np.unique((timestamps[101,:]-timestamps[100,:])[0]))/1000

In [74]:
timestamps[1,0]-timestamps[0,0]

531.995701

In [75]:
f=1/T

In [76]:
f

array([1.87971444])